In [1]:
# Consolidated imports for the whole notebook
import re
import os
import glob
import time
import pandas as pd
import fitz  # PyMuPDF
from google import genai
from google.genai import types

In [2]:
# Source PDF paths (data/) - verified against actual filenames on disk
data_dir = r"C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\data"
pdf_work_law_en = os.path.join(data_dir, "Worklaw_English_1996.pdf")
pdf_work_law_prof_en = os.path.join(data_dir, "Worklaw_English_2019_Professional.pdf")
pdf_work_law_ar = os.path.join(data_dir, "Worklaw_Arabic_1996.pdf")
pdf_work_law_prof_ar = os.path.join(data_dir, "Worklaw_Arabic_2019_Professional.pdf")
pdf_scl_en = os.path.join(data_dir, "SSC_English.pdf")
pdf_scl_ar = os.path.join(data_dir, "SSC_Arabic.pdf")

# Output layout (0. cleaned_data/):
#   OCR/            - raw page-by-page OCR transcriptions (page_NNN.txt per PDF)
#   Text_combined/   - each Arabic source's pages stitched into one .txt file
#   Knowledgebase/   - final cleaned article-level tables (.xlsx)
out_dir = r"C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\0. cleaned_data"
ocr_dir = os.path.join(out_dir, "OCR")
text_combined_dir = os.path.join(out_dir, "Text_combined")
kb_dir = os.path.join(out_dir, "Knowledgebase")
for d in (out_dir, ocr_dir, text_combined_dir, kb_dir):
    os.makedirs(d, exist_ok=True)

In [3]:
os.getcwd()

'c:\\Users\\Mohannad Alsaleh\\Desktop\\Data Science\\Data Science Projects\\1st RAG project\\scripts'

### English File text extraction

#### Work Law + Social Security Law (English)

In [4]:
def get_full_text(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    doc.close()
    return text

def extract_articles_dict_from_text(text, pattern):
    """Split text into per-article chunks keyed by the regex match's captured
    group(s) joined with '-' (e.g. ('85', 'bis') -> '85-bis'). Raises on a
    duplicate key instead of silently overwriting: a duplicate key means two
    headers collapsed onto the same slot, which previously caused silent
    content loss (SCL EN's real Article 85 was overwritten by "Article (85)
    - bis" because both matches captured group(1) == '85')."""
    matches = list(pattern.finditer(text))
    articles = {}
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        key = "-".join(g for g in m.groups() if g)
        if key in articles:
            raise ValueError(
                f"Duplicate article key '{key}' at char {start}: {text[start:start+80]!r}"
            )
        articles[key] = text[start:end].strip()
    return articles

def save_articles_xlsx(articles, source_name, xlsx_name):
    rows = [{"Source": source_name, "Article #": num, "Content": content}
            for num, content in articles.items()]
    df = pd.DataFrame(rows)
    df.to_excel(os.path.join(kb_dir, xlsx_name), index=False, engine="openpyxl")
    print(f"{xlsx_name}: {df.shape} ({source_name})")
    return df

# ---------- WORK LAW EN (1996 body) ----------
# NOTE: Worklaw_English_2019_Professional.pdf is NOT an amendment to specific articles
# of this 1996 law - it is a separate law (Professional Work Regulation Law, itself
# repealing an earlier "Professional Work Regulation Law No. 27 of 1999" - see its
# own Article 10). The two documents' article numbers only coincidentally overlap
# (both start counting from 1, like any standalone law does). An earlier version of
# this pipeline wrongly merged them by matching article number, silently overwriting
# 12 genuine 1996 Labor Law articles with unrelated 2019-law content. Kept separate
# here, matching how the Arabic side already handles it.
pattern_worklaw_base = re.compile(r'^Article\s+(\d+)', re.MULTILINE)
pattern_worklaw_prof = re.compile(r'^Article\s*\((\d+)\)', re.MULTILINE)

worklaw_base_text = get_full_text(pdf_work_law_en)
idx_annex = worklaw_base_text.find("Table No. 1 of 1996")
worklaw_base_text = worklaw_base_text[:idx_annex]
worklaw_base_en = extract_articles_dict_from_text(worklaw_base_text, pattern_worklaw_base)
save_articles_xlsx(worklaw_base_en, "Work_Law_EN_1996", "worklaw_articles_en.xlsx")

worklaw_prof_text = get_full_text(pdf_work_law_prof_en)

# --- known source-PDF gap (Defect 2) ---
# This PDF's own text layer never prints an "Article (12)" header: the clause
# that belongs there ("The Council of Ministers shall issue the necessary
# regulations...") is otherwise silently swallowed into Article (11)'s
# content, and the following article is printed as "Article (13)" - i.e. the
# gap is in the source PDF itself, not an artifact of our extraction. Confirmed
# against worklaw_prof_ar_articles.xlsx: repeal of Law 27/1999 = Arabic
# Article 10 = this English "Article (11)"'s first paragraph; "Council of
# Ministers issues regulations" = Arabic Article 11 = this unlabeled second
# paragraph; "PM/ministers implement" = Arabic Article 12 = English
# "Article (13)". We insert the missing header so the clause splits out as
# its own article instead of merging into 11.
worklaw_prof_text, _n_patched = re.subn(
    r'(?m)^The Council of Ministers shall issue the necessary regulations for the',
    'Article (12) The Council of Ministers shall issue the necessary regulations for the',
    worklaw_prof_text,
    count=1,
)
assert _n_patched == 1, "expected exactly one missing-Article-(12) gap to patch - source PDF text may have changed"

worklaw_prof_en = extract_articles_dict_from_text(worklaw_prof_text, pattern_worklaw_prof)
save_articles_xlsx(worklaw_prof_en, "Professional_Work_Law_EN_2019", "worklaw_prof_articles_en.xlsx")

# ---------- SOCIAL SECURITY LAW EN ----------
# SCL headers are normally "Article (N)"; a small number of later-inserted
# amendment articles are suffixed "- bis" (e.g. "Article (85) – bis"). The
# suffix is captured as a second group so the key becomes "85-bis" instead of
# colliding with plain "85" (Defect 1) - and matches the Arabic file's own
# "85-bis" label so AR<->EN joins work. Real headers stand alone on their own
# line, optionally followed only by a glued-on footnote-marker digit (e.g.
# "Article (85) – bis2  3", "Article (9)1"); mid-sentence cross-references
# that a PDF line-wrap happens to land at the start of a line - e.g.
# "...pursuant to Paragraph (b) of \nArticle (30) of this Law as of..." - have
# real words after them and are rejected by the trailing lookahead instead of
# being mistaken for a second "Article (30)" header (another silent-collision
# case the Defect 1 duplicate-key check surfaced).
pattern_scl = re.compile(r'^Article\s*\((\d+)\)(?:\s*[-–—]\s*(bis))?(?=[ \t\d]*$)', re.MULTILINE)

scl_full_text = get_full_text(pdf_scl_en)
positions_table1 = [m.start() for m in re.finditer(r'Table \(1\)', scl_full_text)]
idx_annex_scl = positions_table1[-1]  # last occurrence = real annex start
scl_body = scl_full_text[:idx_annex_scl]

scl_articles_en = extract_articles_dict_from_text(scl_body, pattern_scl)

# --- Defect 3: strip structural noise (running chapter headers, amendment
# footnotes) that PyMuPDF's reading order interleaves mid-article whenever an
# article spans a page break. Chapter headers are matched against the law's
# fixed 9-chapter table of contents; amendment-provenance footnotes ("As
# amended pursuant to...") are kept, not discarded, in their own column.
SCL_EN_CHAPTER_TITLES = {
    "One": "Chapter One: Definitions and Scopes of Application",
    "Two": "Chapter Two: Administrative Organization of the Corporation",
    "Three": "Chapter Three: Financial Resources of the Corporation",
    "Four": "Chapter Four: Work Injury Insurance",
    "Five": "Chapter Five: Maternity Insurance",
    "Six": "Chapter Six: Unemployment Insurance",
    "Seven": "Chapter Seven: Old-Age, Disability and Death Insurance",
    "Eight": "Chapter Eight: Public Sector Insurance",
    "Nine": "Chapter Nine: General Provisions",
}
SCL_EN_CHAPTER_RE = re.compile(r'^Chapter\s+(' + "|".join(SCL_EN_CHAPTER_TITLES) + r')\b')
# Optional leading digit(s) are a footnote-reference marker glued to the
# front of the footnote line itself (e.g. "2\t As amended pursuant to...").
SCL_EN_AMEND_RE = re.compile(r'^\d{0,2}\s*As amended pursuant to the Law No\.\s*\(\d+\)\s*of\s+\d{4}\s+Amending the Social Security Law\.$')

def clean_scl_en_content(content, current_chapter):
    body_lines, amended = [], []
    for line in content.split("\n"):
        stripped = line.strip()
        chap_m = SCL_EN_CHAPTER_RE.match(stripped)
        if chap_m:
            current_chapter = SCL_EN_CHAPTER_TITLES[chap_m.group(1)]
            continue
        if SCL_EN_AMEND_RE.match(stripped):
            amended.append(stripped)
            continue
        body_lines.append(line)
    return "\n".join(body_lines).strip(), current_chapter, "; ".join(amended)

# Article 1 always opens Chapter One, even though that heading itself sits in
# the (dropped) preamble before Article (1)'s own header.
_current_chapter = SCL_EN_CHAPTER_TITLES["One"]
scl_rows = []
for num, content in scl_articles_en.items():
    cleaned, _current_chapter, amended_by = clean_scl_en_content(content, _current_chapter)
    scl_rows.append({"Source": "Social_Security_Law_EN", "Article #": num, "Content": cleaned,
                      "Chapter": _current_chapter, "Amended By": amended_by})

scl_articles_en_df = pd.DataFrame(scl_rows)
scl_articles_en_df.to_excel(os.path.join(kb_dir, "scl_articles_en.xlsx"), index=False, engine="openpyxl")
print(f"scl_articles_en.xlsx: {scl_articles_en_df.shape} (Social_Security_Law_EN)")
bis = [r for r in scl_rows if "bis" in str(r["Article #"])]
if bis:
    print(f"  note: {len(bis)} bis article(s) found: {[r['Article #'] for r in bis]}")


FileNotFoundError: no such file: 'C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\data\Worklaw_English_1996.pdf'

### Arabic files

#### Stage 1: OCR (page-by-page transcription)

Arabic PDFs here are scanned images, so PyMuPDF text extraction doesn't work — each page is rendered to a PNG and transcribed by a vision model instead. This stage is resumable: if `page_NNN.txt` already exists it's skipped, so re-running this cell after an interruption (or after all pages are already done) costs no API calls. Pages are saved under `cleaned_data/OCR/<name>_pages/`.

In [5]:
# API key must be set outside the notebook, e.g.:
#   PowerShell:  $env:GEMINI_API_KEY = "your-key-here"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def ocr_page_text(img_bytes, max_retries=3):
    prompt = """Transcribe ALL Arabic text from this legal document page, exactly as written, top to bottom, right to left.
Do not translate. Do not summarize. Do not add commentary.
Ignore page numbers, borders, headers/footers, and publisher watermarks.
Output only the raw transcribed text."""
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[types.Part.from_bytes(data=img_bytes, mime_type="image/png"), prompt]
            )
            return resp.text.strip()
        except Exception as e:
            print(f"  retry {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    return ""

def ocr_pdf_to_pages(path, out_pages_dir, dpi=300, delay=4.5):
    os.makedirs(out_pages_dir, exist_ok=True)
    doc = fitz.open(path)
    n = len(doc)
    for i in range(n):
        page_file = os.path.join(out_pages_dir, f"page_{i+1:03d}.txt")
        if os.path.exists(page_file):          # resumable: skip already-done pages
            print(f"Page {i+1}/{n} already done, skipping")
            continue
        pix = doc[i].get_pixmap(dpi=dpi)
        text = ocr_page_text(pix.tobytes("png"))
        with open(page_file, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"Page {i+1}/{n} done ({len(text)} chars)")
        time.sleep(delay)
    doc.close()

KeyError: 'GEMINI_API_KEY'

In [ ]:
# 1. Work Law 1996 (main body, 76 pages)
ocr_pdf_to_pages(pdf_work_law_ar, os.path.join(ocr_dir, "worklaw_ar_pages"))

# 2. Professional Work Regulation Law 2019 (separate law, not an amendment - 3 pages)
ocr_pdf_to_pages(pdf_work_law_prof_ar, os.path.join(ocr_dir, "worklaw_prof_ar_pages"))

# 3. Social Security Law (137 pages)
ocr_pdf_to_pages(pdf_scl_ar, os.path.join(ocr_dir, "scl_ar_pages"))

#### Stage 2: article-level cleaning

A first attempt at splitting the OCR'd pages into articles used a single naive regex on `المادة`. Inspecting the output closely surfaced four real problems that a naive split misses:

1. **OCR digit corruption** — multi-digit Arabic-Indic article numbers sometimes get mangled (e.g. `١٠٢` OCR'd as `١.٢`, or `18`/`19` digit-reversed as `٨١`/`٩١`). Since Jordanian law articles are always strictly sequential with no gaps, the fix is to ignore the OCR'd digit entirely and number articles by counted position instead.
2. **Appended schedules/annexes** — both the Work Law and the Social Security Law have tabular appendices stitched onto the end of the main text (disability-percentage tables, occupational-disease lists, pension-share tables, early-retirement discount tables, etc). These aren't prose articles and their internal numbering restarts from 1, so they must be cut off at a schedule-boundary marker rather than parsed as more articles of the main law. **They are excluded from the final tables for now** — flagged for a dedicated parsing pass later since the numeric tables (especially the Work Law's disability-percentage schedule) have real OCR damage (merged/ambiguous numbers) that needs manual verification before being trusted.
3. **Inline cross-references** — sentences like `...وفق أحكام المادة (٢٩) من هذا القانون.` start with `المادة` mid-sentence and get wrongly matched as a new article header by a naive regex. The fix requires the text *after* the article number to be empty (or just a short footnote/`مكررة` marker) for a match to count as a real header — otherwise it's kept as ordinary body content of the article it actually belongs to.
4. **A genuine inserted amendment article** — the Social Security Law source (current as of 16/4/2023) has an article literally labelled `المادة (٨٥) مكررة` ("Article 85 bis") added by a later amendment that the English translation predates. It's kept as its own `85-bis` row instead of being dropped or silently shifting every subsequent article number out of alignment with the English file. (This is a genuine per-article amendment, unlike the Work Law's 2019 document below, which is a wholly separate law rather than an amendment.)

The cells below re-parse the already-OCR'd page files under `OCR/` with these fixes, write a combined raw-text file per source under `Text_combined/`, and write the final cleaned article tables as `.xlsx` under `Knowledgebase/`.

In [6]:
# ---------- shared helpers ----------

def load_pages_sorted(pages_dir):
    files = sorted(
        glob.glob(os.path.join(pages_dir, "page_*.txt")),
        key=lambda p: int(re.search(r"(\d+)", os.path.basename(p)).group(1)),
    )
    return [open(f, encoding="utf-8").read() for f in files]

# Arabic-Indic (٠-٩) and Extended Arabic-Indic/Persian (۰-۹) digits -> ASCII
_DIGIT_MAP = {ch: str(i) for i, ch in enumerate("٠١٢٣٤٥٦٧٨٩")}
_DIGIT_MAP.update({ch: str(i) for i, ch in enumerate("۰۱۲۳۴۵۶۷۸۹")})

def normalize_digits(s):
    return "".join(_DIGIT_MAP.get(ch, ch) for ch in s)

# Boilerplate lines produced by the OCR/legal-portal watermark, dropped wherever they appear
NOISE_LINES = {
    "الأردن", "الاردن", "قسطاس", "يسر المعرفة",
    "هذه الخاصية معدة بشكل آلي مع بعض المراجعة التحريرية",
    "© قسطاس للتقنيات القانونية",
    "The provided image and its crops consist only of solid color blocks "
    "(white/light gray and dark green). There is no Arabic text present in any of the images.",
    "This image is blank and contains no discernible Arabic text to transcribe.",
}
BARE_DATE_RE = re.compile(r"^[0-9٠-٩۰-۹]{1,4}[/-][0-9٠-٩۰-۹]{1,4}([/-][0-9٠-٩۰-۹]{1,4})?$")
DASH_DIVIDER_RE = re.compile(r"^-{5,}$")

def is_noise(line):
    return line in NOISE_LINES or bool(BARE_DATE_RE.match(line)) or bool(DASH_DIVIDER_RE.match(line))

def write_xlsx(path, header, rows):
    df = pd.DataFrame(rows, columns=header)
    df.to_excel(path, index=False, engine="openpyxl")

#### Work Law Arabic (1996 body + 2019 Professional Work Regulation Law)

`OCR/worklaw_ar_pages/` (76 pages) is the 1996 Labor Law body — it lines up exactly with the English 1996 file: 142 articles, ending with the same Official Gazette footer, followed by 3 appended schedules that are excluded here (see note above).

`OCR/worklaw_prof_ar_pages/` (3 pages) is a separate, shorter law — the 2019 Professional Work Regulation Law (12 articles), itself repealing an earlier 1999 law of the same name. It is **not** an amendment to specific articles of the 1996 Labor Law (its own Article 10 confirms what it repeals), so it's kept as its own file rather than merged into the 1996 one — matching how the English side is now handled too.

Article headers in both sources look like `- المادة ١` or `المادة ٢٦ - عنوان`, optionally followed by a `تاريخ السريان: ...` (effective date) line, which is pulled into its own column.

In [7]:
# Work Law header: line starts with optional dash then "المادة", digits are ignored
# (renumbered sequentially - see OCR digit corruption note above).
WORKLAW_HEADER_RE = re.compile(r"^-?\s*المادة\b\s*[\(\[]?\s*[0-9٠-٩۰-۹.\-\s]*[\)\]]?\s*(.*)$")
EFFDATE_RE = re.compile(r"^تاريخ\s+السريان\s*:?\s*(.+)$")
# Real schedule/appendix boundary marker (not the inline "الجدول رقم (٢) الملحق بهذا القانون" cross-references)
WORKLAW_SCHEDULE_RE = re.compile(r"^جدول\s+رقم\b")
GAZETTE_RE = re.compile(r"^المنشور\s+في\s+العدد\b")

def extract_worklaw_articles(lines):
    """Parse a Work-Law-style segment into (title, content, effective_date) tuples,
    numbered sequentially from 1. Text before the first header (preamble) is dropped."""
    articles = []
    current_title, content_lines, eff_date = None, [], ""

    def flush():
        if current_title is not None:
            articles.append((current_title, "\n".join(content_lines).strip(), eff_date))

    for raw_line in lines:
        line = raw_line.strip()
        if not line or is_noise(line):
            continue
        header_m = WORKLAW_HEADER_RE.match(line)
        if header_m:
            flush()
            current_title, content_lines, eff_date = header_m.group(1).strip(), [], ""
            continue
        if current_title is None:
            continue
        if GAZETTE_RE.match(line):
            break  # publication footer trailing the last article of this segment
        eff_m = EFFDATE_RE.match(line)
        if eff_m:
            eff_date = normalize_digits(eff_m.group(1).strip())
            continue
        content_lines.append(line)
    flush()
    return articles

def process_worklaw_source(pages_dir, combined_txt_name, xlsx_name, source_name, cut_at_first_schedule, text_patches=()):
    pages = load_pages_sorted(pages_dir)
    combined = "\n".join(pages)

    # Apply known, manually-verified OCR corrections (digit misreads etc.) before
    # anything else touches the text, so Text_combined/ and the xlsx both reflect
    # the fix. Each patch asserts its expected occurrence count so a silent
    # source-text change doesn't let a stale/wrong replacement slip through.
    for old, new, expected_count in text_patches:
        actual_count = combined.count(old)
        assert actual_count == expected_count, (
            f"expected {expected_count} occurrence(s) of {old!r} in {combined_txt_name}, "
            f"found {actual_count} - source text may have changed"
        )
        combined = combined.replace(old, new)

    with open(os.path.join(text_combined_dir, combined_txt_name), "w", encoding="utf-8") as f:
        f.write(combined)

    lines = combined.split("\n")
    if cut_at_first_schedule:
        # main law body only - stop at the first appended schedule (appendix excluded for now)
        main_lines = []
        for line in lines:
            if WORKLAW_SCHEDULE_RE.match(line.strip()):
                break
            main_lines.append(line)
        lines = main_lines

    articles = extract_worklaw_articles(lines)
    rows = []
    for i, (title, content, eff_date) in enumerate(articles, start=1):
        # Arabic source -> Arabic header word, matching SCL AR below (not the
        # English "Article" literal - these are two different languages).
        full_content = f"المادة ({i})" + (f": {title}" if title else "")
        if content:
            full_content += f"\n{content}"
        rows.append((source_name, i, full_content, eff_date))

    write_xlsx(os.path.join(kb_dir, xlsx_name), ["Source", "Article #", "Content", "Effective Date"], rows)
    print(f"{xlsx_name}: {len(rows)} rows written ({source_name})")
    return rows

worklaw_ar_rows = process_worklaw_source(
    os.path.join(ocr_dir, "worklaw_ar_pages"), "worklaw_ar_combined.txt", "worklaw_ar_articles.xlsx",
    "Worklaw_Arabic_1996", cut_at_first_schedule=True,
    text_patches=[
        # Article 59(a) overtime rate: OCR mis-transcribed the Arabic-Indic
        # digits as "150%" (duplicating paragraph (b)'s rate) - confirmed
        # against the source scan that it actually reads "126%", consistent
        # with the English translation's "125%" for the same clause.
        ("الاضافية أجرا لا يقل عن ١٥٠% من أجره المعتاد",
         "الاضافية أجرا لا يقل عن ١٢٦% من أجره المعتاد", 1),
    ],
)

worklaw_prof_ar_rows = process_worklaw_source(
    os.path.join(ocr_dir, "worklaw_prof_ar_pages"), "worklaw_prof_ar_combined.txt", "worklaw_prof_ar_articles.xlsx",
    "Professional_Work_Law_AR_2019", cut_at_first_schedule=False)

AssertionError: expected 1 occurrence(s) of 'الاضافية أجرا لا يقل عن ١٥٠% من أجره المعتاد' in worklaw_ar_combined.txt, found 0 - source text may have changed

#### Social Security Law Arabic

`OCR/scl_ar_pages/` (137 pages) — main law body (articles 1-110, plus the `85-bis` amendment article -> 111 physical article headers) followed by 7 appended schedules, excluded here for now (same policy as the Work Law appendix).

Unlike the Work Law, every real article header here is parenthesized (`المادة (٩)`), which makes cross-references like `المادة (٢٩) من هذا القانون.` distinguishable: a real header has nothing (or only a short `مكررة`/footnote marker) after the closing parenthesis, while a cross-reference continues as a full sentence. Two pages were blank/color-only scans where the OCR step returned a vision-model description instead of empty text ("This image is blank...") — those are filtered out as noise.

In [8]:
# SCL headers always parenthesize the number, so a real header requires literal
# parens right after "المادة"; trailing text must be empty or a short "مكررة"
# (bis) / footnote-reference marker for the match to count as a real header.
SCL_HEADER_RE = re.compile(r"^-?\s*المادة\s*[\(\[]\s*[٠-٩۰-۹0-9IV.\-\s]+\s*[\)\]]\s*(.*)$")
SCL_TRAILING_OK_RE = re.compile(r"^(مكررة)?\s*(\(\s*[٠-٩۰-۹0-9]+\s*\))*\s*$")
# Real schedule boundary: a line that is ONLY "(ال)جدول رقم (N)" - distinguishes
# real appendix starts from table-of-contents entries (" : <title>" follows) and
# inline cross-references ("... الملحق بهذا القانون.").
SCL_SCHEDULE_BOUNDARY_RE = re.compile(r"^(?:ال)?جدول\s+رقم\s*[\(\[]\s*[٠-٩۰-۹0-9]+\s*[\)\]]\.?\s*$")

# Defect 3: the law's 9 chapters recur as a bare two-line running header
# ("الفصل التاسع" / "أحكام عامة") stamped at the top of every page within
# that chapter - OCR's page-by-page reading order drops these mid-sentence
# inside whichever article happens to span that page break (Article 86 is a
# clear example: a chapter heading lands between two halves of one sentence).
# Matched as an exact two-line pair (not a loose "starts with الفصل" regex)
# so genuine inline cross-references like "...وفق أحكام المادة (٢٩)..." or the
# front-matter table of contents ("الفصل الأول : ...", single line with a
# colon) are never mistaken for a real chapter boundary.
SCL_AR_CHAPTERS = [
    ("الفصل الأول", "التعاريف ومجال التطبيق"),
    ("الفصل الثاني", "التنظيم الإداري للمؤسسة"),
    ("الفصل الثالث", "الموارد المالية للمؤسسة"),
    ("الفصل الرابع", "تأمين إصابات العمل"),
    ("الفصل الخامس", "تأمين الأمومة"),
    ("الفصل السادس", "تأمين التعطل عن العمل"),
    ("الفصل السابع", "تأمين الشيخوخة والعجز والوفاة"),
    ("الفصل الثامن", "تأمين القطاع العام"),
    ("الفصل التاسع", "أحكام عامة"),
]
# Amendment-provenance footnotes, e.g. "(١) كما هي معدلة بموجب القانون المعدل
# لقانون الضمان الاجتماعي رقم (١١) لسنة ٢٠٢٣." - kept, not discarded, in their
# own column instead of being left splitting the sentence they land in.
SCL_AR_AMEND_FOOTNOTE_RE = re.compile(r"^\(\s*[٠-٩۰-۹0-9]+\s*\)\s*كما\s+هي\s+معدل")

def extract_scl_articles(lines):
    """Parse lines up to the first real schedule boundary into sequential
    articles. Returns list of (article_label, content, chapter, amended_by);
    label is e.g. '85-bis'."""
    articles = []
    current_label, content_lines, counter = None, [], 0
    # Article 1 always opens Chapter One, even though that heading itself
    # sits in the (dropped) preamble before the first real article header.
    current_chapter = f"{SCL_AR_CHAPTERS[0][0]}: {SCL_AR_CHAPTERS[0][1]}"
    amended_by_lines = []

    def flush():
        if current_label is not None:
            articles.append((current_label, "\n".join(content_lines).strip(),
                              current_chapter, "؛ ".join(amended_by_lines)))

    i, n = 0, len(lines)
    while i < n:
        line = lines[i].strip()
        i += 1
        if not line or is_noise(line):
            continue
        if SCL_SCHEDULE_BOUNDARY_RE.match(line):
            break  # main law body ends here; schedules follow
        matched_chapter = False
        for ordinal, title in SCL_AR_CHAPTERS:
            if line == ordinal and i < n and lines[i].strip() == title:
                current_chapter = f"{ordinal}: {title}"
                i += 1  # consume the title line too
                matched_chapter = True
                break
        if matched_chapter:
            continue
        if SCL_AR_AMEND_FOOTNOTE_RE.match(line):
            if current_label is not None:
                amended_by_lines.append(line)
            continue
        header_m = SCL_HEADER_RE.match(line)
        if header_m and SCL_TRAILING_OK_RE.match(header_m.group(1).strip()):
            flush()
            is_bis = "مكررة" in header_m.group(1)
            current_label = f"{counter}-bis" if is_bis else str(counter := counter + 1)
            content_lines, amended_by_lines = [], []
            continue
        if current_label is None:
            continue  # preamble/cover/title-page text before article 1 - dropped
        content_lines.append(line)
    flush()
    return articles

def process_scl_source(pages_dir, combined_txt_name, xlsx_name, source_name):
    pages = load_pages_sorted(pages_dir)
    combined = "\n".join(pages)
    with open(os.path.join(text_combined_dir, combined_txt_name), "w", encoding="utf-8") as f:
        f.write(combined)

    articles = extract_scl_articles(combined.split("\n"))
    rows = []
    for label, content, chapter, amended_by in articles:
        # Arabic source -> Arabic header word (not the English "Article" literal).
        full_content = f"المادة ({label})" + (f"\n{content}" if content else "")
        rows.append((source_name, label, full_content, chapter, amended_by))

    write_xlsx(os.path.join(kb_dir, xlsx_name), ["Source", "Article #", "Content", "Chapter", "Amended By"], rows)
    print(f"{xlsx_name}: {len(rows)} rows written ({source_name})")
    bis = [r for r in rows if "bis" in str(r[1])]
    if bis:
        print(f"  note: {len(bis)} bis article(s) found: {[r[1] for r in bis]}")
    return rows

scl_ar_rows = process_scl_source(
    os.path.join(ocr_dir, "scl_ar_pages"), "scl_ar_combined.txt", "scl_articles_ar.xlsx",
    "Social_Security_Law_AR")


scl_articles_ar.xlsx: 111 rows written (Social_Security_Law_AR)
  note: 1 bis article(s) found: ['85-bis']


#### Chunking

In [9]:
import pandas as pd
from pathlib import Path
KB = Path(r"C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\0. cleaned_data\Knowledgebase")

for f in sorted(KB.glob("*.xlsx")):
    L = pd.read_excel(f)["Content"].str.len()
    print(f"{f.stem:32} n={len(L):3} med={int(L.median()):5} p90={int(L.quantile(.9)):5} max={L.max():5} over3k={int((L>3000).sum())}")

scl_articles_ar                  n=111 med=  546 p90= 2093 max= 9731 over3k=6
scl_articles_en                  n=111 med=  724 p90= 2909 max=13283 over3k=11
worklaw_ar_articles              n=142 med=  420 p90= 1018 max= 3136 over3k=2
worklaw_articles_en              n=142 med=  603 p90= 1484 max= 5727 over3k=3
worklaw_prof_ar_articles         n= 12 med=  153 p90=  471 max=  811 over3k=0
worklaw_prof_articles_en         n= 13 med=  263 p90= 1162 max= 1521 over3k=0


In [10]:
import pandas as pd, re
E = pd.read_excel(KB/"scl_articles_en.xlsx")
row = E.loc[E["Content"].str.len().idxmax()]
print(row["Article #"], len(row["Content"]))
for m in re.finditer(r'^Article\s*\(', row["Content"], re.M):
    print(m.start(), repr(row["Content"][m.start():m.start()+70]))

71 13283
0 'Article (71)1\nFor the purposes of this Chapter:\na. The following words'
3965 'Article (64) of this Law.\n3.\tEarly Retirement Pension shall be increas'


In [11]:
import pandas as pd, re
df = pd.read_excel(KB/"worklaw_ar_articles.xlsx")
long = df[df.Content.str.len() > 2500]
for r in long.itertuples():
    n = len(re.findall(r'^\s*[أ-يـ]{1,3}\s*[-–]\s*', str(r.Content), re.M))
    print(getattr(r,"_2"), len(str(r.Content)), "clause markers:", n)

2 2678 clause markers: 0
12 3114 clause markers: 9
54 3136 clause markers: 9


In [15]:
import pandas as pd, re
from pathlib import Path

KB = Path(r"C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\0. cleaned_data\Knowledgebase")

FILES = {
    "scl_articles_ar.xlsx":          ("ssl",     "ar"),
    "scl_articles_en.xlsx":          ("ssl",     "en"),
    "worklaw_ar_articles.xlsx":      ("labor96", "ar"),
    "worklaw_articles_en.xlsx":      ("labor96", "en"),
    "worklaw_prof_ar_articles.xlsx": ("prof19",  "ar"),
    "worklaw_prof_articles_en.xlsx": ("prof19",  "en"),
}

CLAUSE = {
    "ar": re.compile(r'^\s*([أ-يـ]{1,3})\s*[-–]\s*', re.MULTILINE),
    "en": re.compile(r'^\s*([a-z])\s*[.)]\s*',       re.MULTILINE),
}
MAX_CHARS = 3000

def split_article(text, lang, header):
    marks = list(CLAUSE[lang].finditer(text))
    if len(marks) < 2:
        return [(None, text)]
    stem = text[:marks[0].start()].strip()      # header + intro sentence
    out = []
    for i, m in enumerate(marks):
        end = marks[i+1].start() if i+1 < len(marks) else len(text)
        out.append((m.group(1), f"{stem}\n{text[m.start():end].strip()}"))
    return out

rows = []
for fname, (law, lang) in FILES.items():
    df = pd.read_excel(KB / fname)
    for r in df.itertuples():
        art    = str(getattr(r, "_2"))
        text   = str(r.Content).strip()
        header = text.split("\n")[0]
        pieces = split_article(text, lang, header) if len(text) > MAX_CHARS else [(None, text)]
        for clause, body in pieces:
            rows.append({
                "chunk_id":       f"{law}_{lang}_{art}" + (f"_{clause}" if clause else ""),
                "law":            law,
                "language":       lang,
                "article_number": art,
                "clause":         clause,
                "text":           body,
                "source_file":    fname,
            })

chunks = pd.DataFrame(rows)
chunks.to_parquet(KB.parent / "chunks.parquet", index=False)

print(chunks.shape)
print(chunks.groupby(["law", "language"]).size())
L = chunks["text"].str.len()
print(f"med={int(L.median())} p90={int(L.quantile(.9))} max={L.max()} over3k={int((L>3000).sum())}")

(723, 7)
law      language
labor96  ar          158
         en          143
prof19   ar           12
         en           13
ssl      ar          181
         en          216
dtype: int64
med=464 p90=1379 max=5572 over3k=6


In [17]:
print(chunks[chunks.chunk_id.str.startswith("ssl_en_11_")].text.iloc[2])

Article (11)
The Board shall oversee the Corporation’s affairs and work and shall, for this purpose, be 
vested with all necessary powers and functions, including:
c.	 Monitoring the investment process for the purposes of maximizing the portfolio, hedging, 
following up and achieving the objectives of the investment policy.


In [18]:
p = KB.parent / "chunks.parquet"
print(p, p.exists(), f"{p.stat().st_size/1024:.0f} KB")

C:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project\0. cleaned_data\chunks.parquet True 267 KB
